# Toffoli Gate — Amazon Braket

The Toffoli (CCX) gate flips the target only when **both** controls
are |1\rangle.  With the target starting at |0\rangle, this computes
the classical AND function reversibly.

In [ ]:
import json

from braket.circuit import Circuit
from braket.devices import LocalSimulator

In [ ]:
device = LocalSimulator()

## Toffoli circuit

Qubit 2 = high control, qubit 1 = low control, qubit 0 = target.

In [ ]:
circuit = Circuit()
circuit.ccnot(2, 1, 0)
print(circuit)

## Truth table

In [ ]:
print("c1 c0 t | c1 c0 t'")
print("--------+---------")

for c1 in (0, 1):
    for c0 in (0, 1):
        for t in (0, 1):
            circuit = Circuit()
            if t:
                circuit.x(0)
            if c0:
                circuit.x(1)
            if c1:
                circuit.x(2)
            circuit.ccnot(2, 1, 0)
            result = device.run(circuit, shots=0).result()
            amps = result.result_types[0].value
            out_idx = next(i for i, a in enumerate(amps) if abs(a) > 1e-10)
            out_bits = format(out_idx, '03b')
            print(f" {c1}  {c0}  {t} |  {out_bits[0]}  {out_bits[1]}  {out_bits[2]}")

## Reversible AND

With target at |0\rangle, the Toffoli computes AND:

In [ ]:
for c1, c0 in ((0, 0), (0, 1), (1, 0), (1, 1)):
    circuit = Circuit()
    if c0:
        circuit.x(1)
    if c1:
        circuit.x(2)
    circuit.ccnot(2, 1, 0)
    result = device.run(circuit, shots=0).result()
    amps = result.result_types[0].value
    out_idx = next(i for i, a in enumerate(amps) if abs(a) > 1e-10)
    out_bits = format(out_idx, '03b')
    print(f"  {c1} AND {c0} = {out_bits[2]}   (full ket |{out_bits}>)")

## Controls in superposition

With both controls in equal superposition, the target fires only in |111\rangle.

In [ ]:
circuit = Circuit()
circuit.h(1)
circuit.h(2)
circuit.ccnot(2, 1, 0)
circuit.measure(0)
circuit.measure(1)
circuit.measure(2)
print(circuit)

result = device.run(circuit, shots=2048).result()
counts = result.result_types[0].value
print("counts (bitstring = q2 q1 q0):")
for bits, n in sorted(counts.items()):
    print(f"  |{bits}>  {n}")
print("the target bit is 1 only in |111> — that is the AND of the controls")